# Lab05：從 Anomaly Score 到值得通知的 Incident

核心問題：detector 已經能找到異常後，哪些 signal 真的值得通知人？

本 Lab 不再訓練 detector。Mahalanobis 與 LOF score 已預先產生，我們把注意力放在 production-like monitoring pipeline：狀態、時間、通知政策與可觀測性。Notebook 可離線閱讀；實作時再啟動 Docker Compose，依表格操作 Prometheus、Alertmanager 與 Grafana。

我們會把資料中的異常轉成 Prometheus 告警。

## 1. Detector output 不是 incident

主線是：

`raw anomaly scores → threshold → deduplication → hysteresis → grouping → suppression / inhibition → incident`

- **Prometheus** 把連續 score 轉成 pending／firing／resolved alert state。
- **Alertmanager** 負責 deduplication、grouping、inhibition、Silence、routing 與 notification timing。
- **receiver** 代表真正的 chat／pager destination，保存「實際送出幾次」。
- **Grafana** 只負責觀察，不替 pipeline 做決策。

因此 alert、notification、incident 是三個不同單位。`repeat_interval` 可能讓同一個 alert episode 產生多次 notification；多個 alerts 也可能被合成一次 notification。

## 2. Production-like pipeline 與 production pipeline

本 Lab 從 CSV 檔讀取預先計算好的 scores；真實 production 則由 telemetry ingestion 與線上 detector 持續產生 scores。兩者下游介面一致，所以 alert policy 可以原樣練習。

| Stage | Lab05 | Production counterpart |
| --- | --- | --- |
| Telemetry / detector | CSV replayer | exporter, stream, online detector |
| State evaluation | Prometheus rules | HA rule evaluators |
| Notification policy | Alertmanager | HA Alertmanager cluster / paging platform |
| Receiver | local JSONL webhook | Slack, Teams, email, PagerDuty, ticketing |
| Observation | Grafana | dashboard, SLO, audit and on-call analytics |

Production 還會需要 authentication、TLS、secrets、HA、durable remote storage、backup、RBAC、capacity planning 與 change review；Docker Compose 在此是可重現的單機教學環境，不是假裝已經完成 production hardening。

In [ ]:
from pathlib import Path
import json
import math
import os
from urllib.error import URLError
from urllib.request import urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (11, 4)
pd.set_option("display.max_columns", 20)

fig, ax = plt.subplots(figsize=(13, 3.8))
nodes = [
    (0.5, "CSV\n5-min source data", "#dbeafe"),
    (2.8, "Replayer\n:9200", "#dcfce7"),
    (5.1, "Prometheus\n:9090", "#fee2e2"),
    (7.4, "Alertmanager\n:9093", "#fef3c7"),
    (9.7, "Receiver\n:9300", "#ede9fe"),
]
for x, label, color in nodes:
    box = FancyBboxPatch((x, 1.25), 1.65, 0.9, boxstyle="round,pad=0.08", fc=color, ec="#334155")
    ax.add_patch(box); ax.text(x + 0.825, 1.7, label, ha="center", va="center", fontsize=10)
for left, right in zip(nodes[:-1], nodes[1:]):
    ax.annotate("", xy=(right[0], 1.7), xytext=(left[0] + 1.65, 1.7), arrowprops={"arrowstyle": "->", "lw": 1.8})
ax.text(5.9, 0.45, "Grafana :3000 observes Prometheus metrics and alert state", ha="center", fontsize=11)
ax.annotate("", xy=(5.9, 1.2), xytext=(5.9, 0.65), arrowprops={"arrowstyle": "->", "lw": 1.5})
ax.set(xlim=(0.2, 11.8), ylim=(0.1, 2.6), title="Production-like Alert Pipeline")
ax.axis("off"); plt.tight_layout(); plt.show()

### Docker Compose 在本 Lab 解決什麼？

`infra/lab05/compose.yaml` 一次固定五個服務的版本、container network、host ports、read-only configuration mounts、persistent volumes、health checks 與啟動順序。

- Container 之間用 service name，例如 `prometheus:9090`；瀏覽器才用 `localhost:9090`。
- Prometheus TSDB、Alertmanager state、Grafana state 與 receiver history 使用 named volumes；`docker compose down` 不會刪除它們，除非另加 `-v`。
- active YAML 以 read-only bind mount 提供；學生只修改 repo 裡的檔案，再執行 `check` 與 `reload`。
- image 使用 exact tags，而非 `latest`，讓全班取得一致行為。

In [ ]:
import socket


PORT_SPECS = [
    ("Replayer", "LAB05_REPLAYER_PORT", 9200),
    ("Prometheus", "LAB05_PROMETHEUS_PORT", 9090),
    ("Alertmanager", "LAB05_ALERTMANAGER_PORT", 9093),
    ("Receiver", "LAB05_RECEIVER_PORT", 9300),
    ("Grafana", "LAB05_GRAFANA_PORT", 3000),
]


def find_lab05_dir(start=Path.cwd()):
    """從目前目錄向上尋找 Lab05 Compose 目錄。"""
    for candidate in (start, *start.parents):
        lab05_dir = candidate / "infra" / "lab05"
        if (lab05_dir / "compose.yaml").exists():
            return lab05_dir
    raise FileNotFoundError("找不到 infra/lab05/compose.yaml")


def read_simple_env(path):
    """讀取本 Lab 使用的簡單 KEY=VALUE 設定。"""
    values = {}
    if not path.exists():
        return values
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        key, separator, value = line.partition("=")
        if separator:
            values[key.strip()] = value.strip().strip("\"'")
    return values


def resolve_host_port(variable, default, file_values):
    """依 process environment、.env、預設值的順序解析 port。"""
    raw_value = os.getenv(variable, file_values.get(variable, str(default)))
    try:
        port = int(raw_value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{variable} 必須是 1–65535 的整數，目前是 {raw_value!r}") from exc
    if not 1 <= port <= 65535:
        raise ValueError(f"{variable} 必須是 1–65535 的整數，目前是 {raw_value!r}")
    return port


def tcp_port_is_available(port):
    """嘗試綁定 Docker 會使用的 host address，確認 TCP port 是否可用。"""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
        try:
            probe.bind(("0.0.0.0", port))
        except OSError:
            return False
    return True


lab05_dir = find_lab05_dir()
file_values = read_simple_env(lab05_dir / ".env")
port_rows = []
configuration_errors = []

for service, variable, default in PORT_SPECS:
    try:
        port = resolve_host_port(variable, default, file_values)
    except ValueError as exc:
        port_rows.append({
            "service": service,
            "environment_variable": variable,
            "port": os.getenv(variable, file_values.get(variable, "—")),
            "status": "CONFIG ERROR",
        })
        configuration_errors.append(str(exc))
        continue

    port_rows.append({
        "service": service,
        "environment_variable": variable,
        "port": port,
        "status": "AVAILABLE" if tcp_port_is_available(port) else "IN USE",
    })

port_preflight = pd.DataFrame(port_rows)
display(port_preflight)

conflicts = port_preflight.query("status == 'IN USE'")
if configuration_errors:
    print("⚠️ 請先修正 port 設定：")
    for message in configuration_errors:
        print(f"- {message}")
elif not conflicts.empty:
    ports = ", ".join(str(port) for port in conflicts["port"])
    print(f"⚠️ Host port 已被占用：{ports}")
    print("1. 修改 infra/lab05/.env 的 LAB05_*_PORT，然後重跑本格。")
    print("2. 或先確認占用者，再自行終止適當的 process：")
    print("   Linux/macOS: lsof -nP -iTCP:<PORT> -sTCP:LISTEN")
    print("   Windows PowerShell: Get-NetTCPConnection -LocalPort <PORT>")
    print("若是先前啟動的 Lab05，優先執行 docker compose -f infra/lab05/compose.yaml down。")
else:
    print("✅ Lab05 所需的 host ports 都可用，可以啟動 Docker Compose。")

### Port 衝突時，先找出是誰在管理 process

如果你上週成功啟動本機 Grafana、Prometheus 或 `detector.py`，它們這週可能仍占用 Lab05 Docker 要使用的相同 port。特別是 Grafana：macOS、Linux 與 Windows 的課程安裝方式都把它交給 service manager 管理。直接 `kill` 只處理目前的 PID，沒有告訴 service manager「我要停止服務」；macOS 與 Linux 可能再啟動一個新 process，Windows 是否重啟則取決於 service 的 Recovery 設定。

本格只提供操作指南，**不會替你執行任何停止或終止指令**。先依下表使用原本的管理工具；只有確認 listener 不是 managed service 時，才使用後面的 PID 流程。

#### 先依 port 判斷可能的來源

| Port | Lab05 Docker 角色 | 上週環境最可能的占用者 | 第一個處理方式 |
| ---: | --- | --- | --- |
| `3000` | Grafana | 本機 Grafana service | 使用下一張表中自己平台的「停止並空出 3000」指令 |
| `9090` | Prometheus | 本機 Prometheus | macOS workshop：`brew services stop prometheus`；Linux／Windows 前景執行：回原終端按 `Ctrl+C` |
| `9200` | Replayer | `python labs/workshop/detector.py` | 回原終端按 `Ctrl+C`；找不到終端時才查 PID |
| `9093` | Alertmanager | 上週沒有這個服務 | 若是先前的 Lab05 stack，執行 Compose `down`；否則查 PID |
| `9300` | Receiver | 上週沒有這個服務 | 若是先前的 Lab05 stack，執行 Compose `down`；否則查 PID |

上週的 node_exporter 使用 `9100`，Windows exporter 使用 `9182`；Lab05 沒有 publish 這兩個 ports，因此不必為了本 Lab 停止 exporter。

#### Grafana lifecycle：Lab05 Docker 與上週本機環境

| 環境 | 立即開始 | 登入／開機自啟 | 停止並空出 `3000` | 停用登入／開機自啟 | Lab05 結束後，重新啟用上週環境 |
| --- | --- | --- | --- | --- | --- |
| Lab05 Docker Compose | `docker compose -f infra/lab05/compose.yaml up -d --build` | 本 Lab 未設定；Compose restart policy 預設為 `no` | `docker compose -f infra/lab05/compose.yaml down` | 不需要額外操作 | 再執行 `up -d --build` |
| macOS Homebrew | `brew services run grafana` | `brew services start grafana` | `brew services stop grafana` | 同一個 `stop` 會取消登入自啟 | `brew services start grafana` |
| Linux systemd | `sudo systemctl start grafana-server` | `sudo systemctl enable grafana-server` | `sudo systemctl stop grafana-server` | `sudo systemctl disable grafana-server` | `sudo systemctl enable --now grafana-server` |
| Windows service（系統管理員 PowerShell） | `Start-Service Grafana` | `Set-Service -Name Grafana -StartupType Automatic` | `Stop-Service Grafana` | `Set-Service -Name Grafana -StartupType Manual` | `Set-Service -Name Grafana -StartupType Automatic; Start-Service Grafana` |

macOS 的 `run` 只啟動本次 session；`start` 會啟動並註冊登入自啟，`stop` 則停止並取消註冊。Linux 的 `stop` 與 `disable` 是兩件事：前者立即空出 port，後者只控制下次開機。Windows 的 `Manual` 不會開機自啟，但仍允許手動 `Start-Service`；若設成 `Disabled`，必須先改回 `Manual` 或 `Automatic` 才能啟動。

想先確認狀態，可以執行：

```bash
# macOS
brew services list

# Linux
systemctl status grafana-server
systemctl is-enabled grafana-server
```

Windows 請在 PowerShell 執行：

```powershell
Get-Service Grafana
Get-CimInstance Win32_Service -Filter "Name='Grafana'" |
    Select-Object Name, State, StartMode, ProcessId
sc.exe qfailure Grafana
```

`sc.exe qfailure` 只用來查看 service failure 後是否設定自動 restart；無論結果為何，停止課程安裝的 Grafana 都應使用 `Stop-Service Grafana`，不要直接殺它的 PID。

#### 只有 unmanaged process 才使用 PID 終止流程

如果 listener 不屬於 Homebrew services、systemd、Windows Services 或本 Lab Compose project，才使用以下流程。若不認識或不擁有該 process，請不要終止；改 `infra/lab05/.env` 裡對應的 `LAB05_*_PORT` 會更安全。

Linux／macOS 先找 PID、確認 owner 與 command、儲存尚未保存的工作，再要求正常終止：

```bash
lsof -nP -iTCP:<PORT> -sTCP:LISTEN
ps -p <PID> -o pid=,user=,command=
kill <PID>
lsof -nP -iTCP:<PORT> -sTCP:LISTEN
```

等候片刻並重查 port。只有正常終止失敗，而且再次確認 PID 沒有改變時，才把 `kill -9 <PID>` 當成**最後手段**。

Windows PowerShell 先檢查 connection 與 process；如果回傳多筆 listener，請逐筆確認，不要整批終止：

```powershell
$connection = Get-NetTCPConnection -LocalPort <PORT> -State Listen
$connection
Get-Process -Id $connection.OwningProcess
Stop-Process -Id $connection.OwningProcess
Get-NetTCPConnection -LocalPort <PORT> -State Listen -ErrorAction SilentlyContinue
```

只有正常終止失敗，而且再次確認 process 正確時，才把 `Stop-Process -Id $connection.OwningProcess -Force` 當成**最後手段**。

完成後重跑上一格；全部顯示 `AVAILABLE` 再啟動 Lab05 Docker Compose。

啟動指令：

```bash
docker compose -f infra/lab05/compose.yaml up -d --build
python labs/workshop/lab05_control.py health
```

In [ ]:
def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "data" / "synthetic" / "lab05_replay_metrics.csv").exists():
            return candidate
    raise FileNotFoundError("Cannot locate repository root")

ROOT = find_repo_root()
DATA = ROOT / "data" / "synthetic"
metrics = pd.read_csv(DATA / "lab05_replay_metrics.csv", parse_dates=["source_timestamp"])
events = pd.read_csv(DATA / "lab05_event_catalog.csv", parse_dates=["start_time", "end_time"])

SOURCE_CADENCE_SECONDS = 300
THRESHOLD = 3.0
SCENARIOS = ["P1", "P2", "P3", "P4", "full_incident"]

summary = (metrics.groupby("scenario_id")
           .agg(points=("sample_index", "nunique"), targets=("target", "nunique"),
                start=("source_timestamp", "min"), end=("source_timestamp", "max"))
           .reindex(SCENARIOS))
display(summary)

In [ ]:
fig, axes = plt.subplots(len(SCENARIOS), 1, figsize=(13, 12), sharex=False)
for ax, scenario in zip(axes, SCENARIOS):
    part = metrics.query("scenario_id == @scenario and target == 'fw01'")
    ax.plot(part["sample_index"], part["mahalanobis_score"], label="Mahalanobis", lw=1.8)
    ax.plot(part["sample_index"], part["lof_score"], label="LOF", lw=1.2, alpha=.8)
    ax.axhline(THRESHOLD, color="black", ls="--", lw=1, label="Threshold" if scenario == "P1" else None)
    ax.set(title=scenario, ylabel="Score", ylim=(0, 6.5))
axes[0].legend(ncol=3, loc="upper right")
axes[-1].set_xlabel("Five-minute source sample index")
fig.suptitle("Replay Scenarios", y=1.01, fontsize=14)
plt.tight_layout(); plt.show()

## 3. 雙時鐘：保留五分鐘語意，但不用真的等五分鐘

資料的 **source clock** 永遠是五分鐘一點：$\Delta t_{source}=300\text{ s}$。Replayer 的 **wall/replay clock** 用可調速度 $S$ 加速：

$$
\Delta t_{replay}=\frac{\Delta t_{source}}{S}=\frac{300}{S}\text{ s}
$$

若來源事件持續 $D_{source}$ 秒，課堂上會播放：

$$
D_{replay}=\frac{D_{source}}{S}
$$

預設 $S=300$，所以一個五分鐘 source point 播一秒。最大 $S=600$，使每點仍維持 $0.5$ 秒，至少讓 250 ms scrape 看見兩次；再快就可能整點跳過。

In [ ]:
def replay_interval_seconds(source_cadence_seconds, replay_speed):
    if not 0 < replay_speed <= 600:
        raise ValueError("replay_speed must be in (0, 600]")
    return source_cadence_seconds / replay_speed

def source_duration_to_lab_seconds(source_seconds, replay_speed):
    return source_seconds / replay_speed

def lab_duration_to_source_seconds(lab_seconds, replay_speed):
    return lab_seconds * replay_speed

assert replay_interval_seconds(300, 300) == 1

In [ ]:
timer_table = pd.DataFrame([
    {"Replay speed S": speed,
     "Seconds per source point": replay_interval_seconds(300, speed),
     "30 source minutes take (s)": source_duration_to_lab_seconds(30 * 60, speed),
     "for: 2s means source time (min)": lab_duration_to_source_seconds(2, speed) / 60}
    for speed in (1, 150, 300, 600)
])
display(timer_table)

## 4. 超過 Threshold，不代表立刻通知

Threshold 只把 score $z_t$ 轉成 raw condition $c_t=\mathbb{1}(z_t>h)$。真正走到通知的路徑是：`score → threshold condition → Prometheus state → Alertmanager`。

Prometheus alert state 依序是 `inactive → pending → firing`。沒有 `for:` 時，condition 成立便可進入 firing；有 `for: d` 時，condition 必須持續成立 $d$ 秒才會從 pending 進入 firing。`keep_firing_for: k` 則在 condition 恢復後暫時保留 firing。

這三種穩定化方法處理的是不同問題：

| 方法 | 記憶的是什麼 | 用途 |
| --- | --- | --- |
| `for:` | condition 成立了多久 | 過濾太短的 spike |
| `keep_firing_for:` | condition 恢復了多久 | 跨過短暫恢復，減少 firing／resolved 反覆切換 |
| two-threshold hysteresis | 前一刻在正常或異常狀態 | 以 $h_{on}>h_{off}$ 建立 amplitude deadband |

因此 `keep_firing_for:` 是時間記憶，不等於 true hysteresis。下一節先定義共同實驗流程，再由 P1 實際修改設定並比較前後差異。

In [ ]:
def simulate_alert_state(flags, for_points=1, keep_points=0):
    flags = np.asarray(flags, dtype=bool)
    states, run, remaining = [], 0, 0
    firing = False
    for flag in flags:
        run = run + 1 if flag else 0
        if run >= for_points:
            firing, remaining = True, keep_points
        elif firing and not flag and remaining > 0:
            remaining -= 1
        elif not flag or run < for_points:
            firing = False
        states.append("firing" if firing else ("pending" if flag else "inactive"))
    return np.array(states)

toy_flags = np.array([0, 0, 1, 0, 1, 1, 1, 0, 1, 0], dtype=bool)
toy_states = simulate_alert_state(toy_flags, for_points=3, keep_points=2)
mapping = {"inactive": 0, "pending": 1, "firing": 2}
fig, ax = plt.subplots(figsize=(10, 2.8))
ax.step(range(len(toy_states)), [mapping[s] for s in toy_states], where="post")
ax.set(yticks=[0, 1, 2], yticklabels=["inactive", "pending", "firing"],
       xlabel="Evaluation step", title="Alert State")
plt.tight_layout(); plt.show()

In [ ]:
# 啟動 Compose 後執行；使用與 port preflight 相同的設定來源。
HEALTH_PATHS = {
    "Replayer": "/-/healthy",
    "Prometheus": "/-/healthy",
    "Alertmanager": "/-/healthy",
    "Receiver": "/-/healthy",
    "Grafana": "/api/health",
}

file_values = read_simple_env(find_lab05_dir() / ".env")
resolved_ports = {
    service: resolve_host_port(variable, default, file_values)
    for service, variable, default in PORT_SPECS
}
endpoints = {
    service: f"http://localhost:{resolved_ports[service]}{path}"
    for service, path in HEALTH_PATHS.items()
}
rows = []
for component, url in endpoints.items():
    try:
        with urlopen(url, timeout=2) as response:
            rows.append({"Component": component, "Status": "PASS" if response.status == 200 else "FAIL", "Action": url})
    except (URLError, OSError) as error:
        rows.append({"Component": component, "Status": "FAIL", "Action": f"Check {url}: {error}"})
preflight = pd.DataFrame(rows)
display(preflight)

## 5. 實驗方式：P1–P3 比較修改前與修改後

P1–P3 是**必做且累積式**的 live 實驗；P4 是選做延伸。開始 P1 前只還原一次課程基線，並讓 running services 載入它：

```bash
python labs/workshop/lab05_control.py restore-config
python labs/workshop/lab05_control.py reload
```

之後 P1–P3 都使用同一個 before／edit／after 循環：

```bash
# 修改前：播放該 scenario，記下 start 印出的 run_id
python labs/workshop/lab05_control.py reset
python labs/workshop/lab05_control.py start <P1|P2|P3> --speed 300
python labs/workshop/lab05_control.py status

# 只修改該 scenario 指定的檔案，然後執行：
python labs/workshop/lab05_control.py check
python labs/workshop/lab05_control.py reload

# 修改後：重播同一 scenario，記下新的 run_id
python labs/workshop/lab05_control.py reset
python labs/workshop/lab05_control.py start <P1|P2|P3> --speed 300
python labs/workshop/lab05_control.py status
```

如果 `status` 的 `state` 還不是 `complete`，稍後再執行一次。到 Grafana 用 `Run` filter 分別選修改前與修改後的 `run_id`，並查看該 run 的 Receiver payload：

```text
http://localhost:9300/api/notifications?run_id=<run_id>
```

若 `.env` 改過 Receiver host port，請替換 `9300`。Receiver history 與 Grafana 的 delivery counters 都是 append-only；`reset` 只重設 replayer，不會清除舊資料，所以不能把 `firing - resolved` 當成 active alerts 數量。

> **不要在 P1 到 P3 之間執行 `restore-config`。** 每個 Checkpoint 都要保留。Notebook 的 Python 圖只解釋靜態 CSV 裡的 source signals；圖畫成功不代表 live Prometheus／Alertmanager 已載入你修改的 YAML。


## 6. P1 — 短 spike、persistence 與 flapping

### 這一節要學什麼

實際操作 `for:` 與 `keep_firing_for:`：前者要求 condition 持續一段時間才 firing，後者在 condition 短暫恢復時延後 resolved。這兩個 timer 都使用 replay wall time；`S=300` 時，`2s` 相當於 10 分鐘 source time。

### 修改前：先跑一次

```bash
python labs/workshop/lab05_control.py reset
python labs/workshop/lab05_control.py start P1 --speed 300
python labs/workshop/lab05_control.py status
```

記下 baseline `run_id`，等 `state: complete` 後觀察 Grafana `Alert State` 與該 run 的 Receiver history。

### 修改哪個檔案

修改 `infra/lab05/prometheus/rules.yml` 的 `HighAnomalyScore`。找到 `# BEGIN LAB EDIT P1` 與 `# END LAB EDIT P1`；這是 Compose 掛載進 Prometheus 的 active rules file。

### 實際修改內容

把以下兩行放進 marker，縮排要和 `expr`、`labels`、`annotations` 同層：

```yaml
        # BEGIN LAB EDIT P1
        for: 2s
        keep_firing_for: 2s
        # END LAB EDIT P1
```

選 `2s` 是因為單點 spike 撐不到兩秒，而 persistent segment 可以完成 pending；短暫低點則由 `keep_firing_for` 跨過。

### 套用並重跑

```bash
python labs/workshop/lab05_control.py check
python labs/workshop/lab05_control.py reload
python labs/workshop/lab05_control.py reset
python labs/workshop/lab05_control.py start P1 --speed 300
python labs/workshop/lab05_control.py status
```

記下新的 `run_id`，並等 `state: complete`。

### 修改前／後應該看到什麼

| P1 訊號 | 修改前 | 修改後 |
| --- | --- | --- |
| 單點 spike | threshold crossing 可直接 firing／通知 | 只到 pending，不完成 `for: 2s`，不應有獨立 firing notification |
| persistent segment | firing | 先 pending，再 firing |
| firing 中的短暫低點 | 容易 resolved 後再 firing | `keep_firing_for: 2s` 暫時保留 firing |
| live evidence | Grafana `Alert State` 與 baseline `run_id` 的 Receiver payload | 用 after `run_id` 看相同位置並比較 |

Grafana `Notifications` 的 `deliveries` 是 Receiver 收到的 webhook 累積次數，不是 active alerts。`repeat_interval` 也可能讓同一個 firing episode 送出多次 delivery，因此本實驗**不要求固定的 firing／resolved 總數**；判讀重點是短 spike 沒有獨立通知，而 persistent segment 有進入 firing。

下方 Python 圖是靜態資料的離線 state 模擬，不會查詢 live Prometheus，也不能取代上述 `run_id` 比較。

### Checkpoint

保留 `for: 2s` 與 `keep_firing_for: 2s`。**不要 restore**；P2 會在此基礎上繼續修改 Alertmanager。


In [ ]:
p1 = metrics.query("scenario_id == 'P1' and target == 'fw01'").reset_index(drop=True)
p1_flags = p1["mahalanobis_score"].gt(THRESHOLD).to_numpy()
interval = replay_interval_seconds(300, 300)
for_points = math.ceil(2 / interval) + 1
p1_states = simulate_alert_state(p1_flags, for_points=for_points, keep_points=2)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(p1["sample_index"], p1["mahalanobis_score"], marker="o", ms=3)
axes[0].axhline(THRESHOLD, color="black", ls="--")
axes[0].set(title="P1 Persistence", ylabel="Score")
axes[1].step(p1["sample_index"], [{"inactive": 0, "pending": 1, "firing": 2}[s] for s in p1_states], where="post")
axes[1].set(yticks=[0, 1, 2], yticklabels=["inactive", "pending", "firing"], xlabel="Five-minute source sample")
plt.tight_layout(); plt.show()
print({"spike_state": p1_states[5], "persistent_fires": bool((p1_states[9:14] == "firing").any())})

### 選做延伸：true hysteresis 是 detector 操作，不是 Prometheus timer

`keep_firing_for` 只能提供時間記憶。真正的 two-threshold hysteresis 以 $h_{on}>h_{off}$ 建立 amplitude deadband：超過 $h_{on}$ 才進入異常，降到 $h_{off}$ 以下才離開，中間則維持上一個狀態。

本 Lab 沒有可改的 `hysteresis.yml`；live replayer 只曝露預先計算好的 score。以下程式是 notebook 裡的 offline 比較，不會改變 Compose。`h_on`、`h_off` 也不是 Prometheus rule keys，不能加進 `rules.yml`：

```python
def apply_hysteresis(scores, h_on=3.0, h_off=2.0):
    if h_on <= h_off:
        raise ValueError("h_on must be greater than h_off")
    firing = False
    states = []
    for score in scores:
        if not firing and score > h_on:
            firing = True
        elif firing and score < h_off:
            firing = False
        states.append(firing)
    return np.asarray(states, dtype=bool)

p1_scores = (
    metrics.query("scenario_id == 'P1' and target == 'fw01'")
    .sort_values("sample_index")["mahalanobis_score"]
    .to_numpy()
)
p1_hysteresis = apply_hysteresis(p1_scores, h_on=3.0, h_off=2.0)
pd.DataFrame({"score": p1_scores, "hysteresis_firing": p1_hysteresis})
```

若要把 hysteresis 做進 live pipeline，才需要新增 stateful detector 或修改 replayer，使它輸出保存狀態後的 metric；這不是 P1 必做設定。

## 7. P2 — 三個 alerts 是否應該變三封通知？

### 這一節要學什麼

操作 Alertmanager grouping 與 notification timers。Deduplication 辨識同一 alert instance；grouping 則把多個 alerts 放進同一通知群組。通知次數不等於事件數。P1 的設定繼續保留。

### 修改前：先跑一次

```bash
python labs/workshop/lab05_control.py reset
python labs/workshop/lab05_control.py start P2 --speed 300
python labs/workshop/lab05_control.py status
```

記下 baseline `run_id`。Baseline 依 `alertname, target, detector, run_id` 分組，因此同一 site 的三個 alert identities 會落在不同 groups。

### 修改哪個檔案

修改 `infra/lab05/alertmanager/alertmanager.yml` 最上方的 root `route`，位置就在 `# BEGIN LAB EDIT P2/P4` 前。

### 實際修改內容

把 root route 原本四個欄位改成：

```yaml
route:
  receiver: chat
  group_by: [site, run_id]
  group_wait: 2s
  group_interval: 3s
  repeat_interval: 6s
  # BEGIN LAB EDIT P2/P4
```

`site, run_id` 把同次 replay 的相關 alerts 放進同一 group；三個 timer 分別控制第一封等待、group 更新與重送。

### 套用並重跑

```bash
python labs/workshop/lab05_control.py check
python labs/workshop/lab05_control.py reload
python labs/workshop/lab05_control.py reset
python labs/workshop/lab05_control.py start P2 --speed 300
python labs/workshop/lab05_control.py status
```

記下新的 `run_id`，並等 `state: complete`。分別打開 before／after payload：

```text
http://localhost:9300/api/notifications?run_id=<run_id>
```

若 `.env` 改過 Receiver host port，請替換 `9300`。

### 修改前／後應該看到什麼

| 觀察 | 修改前 | 修改後 |
| --- | --- | --- |
| Prometheus alerts | 三個 alert identities | 仍是三個，grouping 不會刪 alert |
| Receiver `groupKey` | 三個不同 groups | 同一個 `site/run_id` group |
| payload 的 `alerts` array | delivery 通常各自帶一個 alert identity | 同一 delivery 可包含多個 alerts；較晚出現的 alert 也會更新同一 group |
| delivery 次數 | 多個獨立 group deliveries | 仍可能因 group update 或 repeat 超過一封 |

P2 的 signals 在 samples 6、8、10 出現；`S=300` 時橫跨 4 秒，所以 `group_wait: 2s` **不保證第一個 webhook 已包含全部三個 alerts**。比較 `groupKey` 與每封 webhook 的 `alerts`，不要只比較總封數。

下方 Python 圖與表格只顯示靜態 CSV 的 signal onset 與 grouping opportunity，不是從 Alertmanager 實測的 webhook 結果。

### Checkpoint

保留 P1 timers 與本節四個 root-route 值。**不要 restore**；P3 會再加入 inhibition。


In [ ]:
p2 = metrics.query("scenario_id == 'P2' and target == 'fw01'").reset_index(drop=True)
signals = pd.DataFrame({
    "Mahalanobis": p2["mahalanobis_score"].gt(THRESHOLD),
    "LOF": p2["lof_score"].gt(THRESHOLD),
    "Packet loss": p2["packet_loss_ratio"].gt(.02),
})
onsets = signals & ~signals.shift(fill_value=False)
p2_table = pd.DataFrame({
    "Policy": ["Separate by alertname/target/detector", "Group by site/run_id"],
    "First-dispatch groups": [int(onsets.sum().sum()), 1],
    "Alerts represented": [int(signals.any().sum()), int(signals.any().sum())],
})
display(p2_table)

fig, ax = plt.subplots(figsize=(11, 3.5))
for offset, column in enumerate(signals):
    ax.step(p2["sample_index"], signals[column].astype(int) + offset * 1.3, where="post", label=column)
ax.set(title="P2 Grouping Opportunity", xlabel="Five-minute source sample", yticks=[])
ax.legend(ncol=3); plt.tight_layout(); plt.show()

## 8. P3 — 根因出現後，症狀 alert 仍存在但不通知

### 這一節要學什麼

操作 inhibition：當同一 site、同一 run 的 `FirewallDown` 根因存在時，不再把 `WebDown`、`DatabaseDown` 症狀通知給人。P1、P2 設定繼續保留。

### 修改前：先跑一次

```bash
python labs/workshop/lab05_control.py reset
python labs/workshop/lab05_control.py start P3 --speed 300
python labs/workshop/lab05_control.py status
```

記下 baseline `run_id`。Prometheus 與 Receiver history 應可看到 Firewall、Web、Database 三種 service alerts。

### 修改哪個檔案

修改 `infra/lab05/alertmanager/alertmanager.yml` 的 top-level `# BEGIN LAB EDIT P3` 到 `# END LAB EDIT P3`。`inhibit_rules` 和 `route`、`receivers` 同層。

### 實際修改內容

```yaml
# BEGIN LAB EDIT P3
inhibit_rules:
  - source_matchers:
      - 'alertname="FirewallDown"'
    target_matchers:
      - 'alertname=~"WebDown|DatabaseDown"'
    equal: [site, run_id]
# END LAB EDIT P3
```

`equal` 防止別的 site 或別次 replay 的根因誤抑制目前症狀。

### 套用並重跑

```bash
python labs/workshop/lab05_control.py check
python labs/workshop/lab05_control.py reload
python labs/workshop/lab05_control.py reset
python labs/workshop/lab05_control.py start P3 --speed 300
python labs/workshop/lab05_control.py status
```

記下新的 `run_id`，並等 `state: complete`；再用 Grafana 與 `/api/notifications?run_id=<run_id>` 比較 live 結果。

### 修改前／後應該看到什麼

| 觀察位置 | 修改前 | 修改後 |
| --- | --- | --- |
| Grafana `Alert State`／Prometheus | FirewallDown、WebDown、DatabaseDown 都 firing | 三者仍 firing，保留 diagnosis／audit 證據 |
| Receiver `status: firing` payload | 三種 service alert names 都可能送達 | 不應包含 `WebDown` 或 `DatabaseDown`；它們被 inhibition |
| 最後一封 `status: resolved` payload | 包含已 resolved 的 group members | 仍可能各包含一次 `WebDown`、`DatabaseDown` |

成功條件是 edited run 的所有 `status: firing` payload 都沒有 `WebDown`、`DatabaseDown`，不是要求 Receiver 只能看到 `FirewallDown`。P3 合成資料也讓 fw01、web01、db01 的 anomaly score 超過 threshold，所以不在本條 inhibition target 裡的 `HighAnomalyScore` 仍可能送達。

當 `FirewallDown` 也 resolved 時，source alert 不再存在，inhibition 隨之結束；而 `chat` webhook 設了 `send_resolved: true`，所以最後一封 resolved group notification 仍可能各帶一次已 resolved 的 `WebDown`、`DatabaseDown`。這不代表 firing notification 洩漏。

下方 Python 圖只畫三個 service 的 source-time up/down 訊號；summary table 是政策預期，不是 live Receiver 查詢。

### Checkpoint

保留 P1 timers、P2 grouping/timers 與本節 `inhibit_rules`。必做路徑已完成；不要 restore，下一節 P4 可選做，也可以直接進 Section 10。


In [ ]:
p3 = metrics.query("scenario_id == 'P3'")
service = p3.pivot(index="sample_index", columns="target", values="service_up")
fig, ax = plt.subplots(figsize=(11, 3.5))
for offset, target in enumerate(("fw01", "web01", "db01")):
    ax.step(service.index, (1 - service[target]) + offset * 1.3, where="post", label=target)
ax.set(title="P3 Root Cause and Symptoms", xlabel="Five-minute source sample", yticks=[])
ax.legend(ncol=3); plt.tight_layout(); plt.show()
with pd.option_context("display.max_colwidth", None, "display.width", 160):
    display(pd.DataFrame({
        "Layer": ["Prometheus firing alerts", "Receiver after inhibition"],
        "Visible names": [
            "FirewallDown, WebDown, DatabaseDown",
            "Firing: no WebDown / DatabaseDown; resolved may include both",
        ],
    }))


## 9. 選做 P4 — Context-aware routing：maintenance 不通知，critical 才送 pager

### 這一節要學什麼

P1–P3 是必做路徑；本節是選做延伸。若跳過 P4，可保持 P3 checkpoint 並直接前往 Section 10。

操作 label-driven routing：`maintenance="true"` 送到不含 integration 的 `maintenance-sink`；其他 alerts 才依 severity 送到 `chat` 或 `pager`。這讓非固定、非週期的維護狀態由上游 context 決定，policy 則留在版本控管的 Alertmanager config。P1–P3 設定繼續保留。

### 修改前：先跑一次

```bash
python labs/workshop/lab05_control.py reset
python labs/workshop/lab05_control.py start P4 --speed 300
python labs/workshop/lab05_control.py status
```

記下 baseline `run_id`。沒有 child routes 時，所有 alerts 都沿 root route 送到 `chat`，包括 critical 與 maintenance。

### 修改哪個檔案

修改 `infra/lab05/alertmanager/alertmanager.yml` 的兩個位置：

1. 在 root route 的 `# BEGIN LAB EDIT P2/P4` marker 內加入 ordered child routes。
2. 在 `pager` receiver 後的 `# BEGIN LAB EDIT P4 RECEIVER` marker 內加入 `maintenance-sink`。

### 實際修改內容

先在 `# BEGIN LAB EDIT P2/P4` 與 `# END LAB EDIT P2/P4` 之間加入：

```yaml
  routes:
    - receiver: maintenance-sink
      matchers:
        - 'maintenance="true"'
    - receiver: pager
      matchers:
        - 'severity="critical"'
    - receiver: chat
      matchers:
        - 'severity="warning"'
```

再到 `receivers:` 清單，在 `pager` 後加入：

```yaml
  # BEGIN LAB EDIT P4 RECEIVER
  - name: maintenance-sink
  # END LAB EDIT P4 RECEIVER
```

順序是 policy 的一部分：預設不使用 `continue: true` 時，第一個符合的 child route 會停止往後比對。因此 maintenance route 必須放在 severity routes 前面，否則 maintenance critical 會先送到 pager。

### 套用並重跑

```bash
python labs/workshop/lab05_control.py check
python labs/workshop/lab05_control.py reload
python labs/workshop/lab05_control.py reset
python labs/workshop/lab05_control.py start P4 --speed 300
python labs/workshop/lab05_control.py status
```

記下新的 `run_id`，並等 `state: complete`。

### 修改前／後應該看到什麼

| Context | 修改前 | 修改後 |
| --- | --- | --- |
| normal warning | `chat` | `chat` |
| normal critical | `chat` | `pager` |
| maintenance warning／critical | `chat` | `maintenance-sink`，不對外送 notification |
| Prometheus alerts | 仍可見 | 仍可見；routing 只改通知去向 |

> **Production 對照：** Silence API／`amtool` 適合上游沒有 context label 時的臨時例外；固定日曆排程可用 `time_intervals`。本實驗已有 `maintenance` label，因此必做操作是 config-based routing，不需要在 Web UI 手動建立 Silence。

### Checkpoint

若完成本選做，保留 P1–P4 設定；若跳過，保留 P1–P3。**不要 restore**；下一節會把 P1–P3 串回同一個 incident，並另外標示 P4 的選做觀察。

In [ ]:
p4 = metrics.query("scenario_id == 'P4' and target == 'fw01'").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(p4["sample_index"], p4["mahalanobis_score"], label="Warning score")
ax.plot(p4["sample_index"], p4["packet_loss_ratio"] * 100, label="Packet loss (%)")
ax.fill_between(p4["sample_index"], 0, 6, where=p4["maintenance_active"].astype(bool), alpha=.18, label="Maintenance")
ax.axhline(THRESHOLD, color="black", ls="--", lw=1)
ax.set(title="P4 Context and Routing", xlabel="Five-minute source sample", ylabel="Value", ylim=(0, 6))
ax.legend(ncol=3); plt.tight_layout(); plt.show()
display(pd.DataFrame([
    {"Context": "Normal warning", "Alert visible": True, "Receiver": "chat"},
    {"Context": "Normal critical", "Alert visible": True, "Receiver": "pager"},
    {"Context": "Maintenance", "Alert visible": True, "Receiver": "maintenance-sink"},
]))

## 10. Full incident — 把 P1–P3 policy 串回同一條 timeline

**不要 restore。** 必做路徑使用 P1–P3 累積完成的設定。開始前確認：

1. `HighAnomalyScore` 有 `for: 2s` 與 `keep_firing_for: 2s`。
2. root `group_by` 是 `[site, run_id]`。
3. `group_wait`／`group_interval`／`repeat_interval` 是 `2s`／`3s`／`6s`。
4. `inhibit_rules` 以 `FirewallDown` 抑制同 site/run 的 `WebDown`、`DatabaseDown`。

完整情境依序包含 nuisance spike、persistent score、flapping、multi-signal、firewall root cause 與 maintenance。執行後記下同一個 `run_id`，按以下順序解讀 live evidence：

1. Grafana `Anomaly Scores`／`Alert State`：短 spike 不 firing，persistent score 先 pending 再 firing。
2. Receiver payload：相關 alerts 使用相同的 `site/run_id` `groupKey`；一封 delivery 可以包含多個 `alerts`。
3. Grafana／Prometheus：`FirewallDown`、`WebDown`、`DatabaseDown` 仍可見。
4. Receiver `status: firing` payload：`FirewallDown` 存在時，不應含 `WebDown` 或 `DatabaseDown`；其他如 `HighAnomalyScore` 仍可能合法送達。
5. 最後一封 `status: resolved` payload：因 `send_resolved: true`，可能各包含一次已 resolved 的 `WebDown`、`DatabaseDown`。

```bash
python labs/workshop/lab05_control.py reset
python labs/workshop/lab05_control.py start full_incident --speed 300
python labs/workshop/lab05_control.py status
```

等 `state: complete` 後，查看：

```text
http://localhost:9300/api/notifications?run_id=<run_id>
```

> **選做 P4 的額外觀察：** 若已完成 P4，normal warning 到 `chat`、normal critical 到 `pager`，maintenance-labelled alerts 進 `maintenance-sink` 而沒有 webhook。若跳過 P4，critical 與 maintenance-labelled alerts 繼續沿 root route 到 `chat`，這是預期結果，不算 P1–P3 驗證失敗。


In [ ]:
# 這裡只讀靜態 CSV 的 fw01 source signals，不會查詢本次 live run_id。
# 圖用來定位訊號階段；實際 alert／notification 結果仍以 Grafana 與 Receiver payload 為準。
full = metrics.query("scenario_id == 'full_incident' and target == 'fw01'").reset_index(drop=True)
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
axes[0].plot(full["sample_index"], full["mahalanobis_score"], label="Mahalanobis")
axes[0].plot(full["sample_index"], full["lof_score"], label="LOF")
axes[0].axhline(THRESHOLD, color="black", ls="--", label="Threshold")
axes[0].set(title="Full Incident Replay", ylabel="Score"); axes[0].legend(ncol=3)
axes[1].plot(full["sample_index"], full["packet_loss_ratio"] * 100, label="Packet loss (%)")
axes[1].step(full["sample_index"], (1 - full["service_up"]) * 4, where="post", label="Firewall down")
axes[1].fill_between(full["sample_index"], 0, 6, where=full["maintenance_active"].astype(bool), alpha=.18, label="Maintenance")
axes[1].set(xlabel="Five-minute source sample", ylabel="Operational signal"); axes[1].legend(ncol=3)
plt.tight_layout(); plt.show()

In [ ]:
scorecard = pd.DataFrame([
    {"Scenario": "P1", "Scope": "Required", "Decision": "for + keep_firing_for", "Observe": "Alert State + Receiver payload"},
    {"Scenario": "P2", "Scope": "Required", "Decision": "grouping + notification timers", "Observe": "groupKey + contained alerts"},
    {"Scenario": "P3", "Scope": "Required", "Decision": "inhibition", "Observe": "firing inhibited; resolved may include symptoms"},
    {"Scenario": "P4", "Scope": "Optional", "Decision": "routing + label suppression", "Observe": "receiver and maintenance context"},
    {"Scenario": "Full", "Scope": "Required integration (P1–P3)", "Decision": "combine required policies", "Observe": "one run_id across the live timeline"},
])
with pd.option_context("display.max_columns", None, "display.width", 160):
    display(scorecard)


## 11. 結論：值得叫醒人的不是每一次 threshold crossing

1. P1 用 `for:` 過濾短暫 condition，用 `keep_firing_for:` 降低 state flapping；後者不是 two-threshold hysteresis。
2. P2 的 grouping 改變 delivery 的 `groupKey` 與 payload 形狀，不會刪除 Prometheus alerts，也不保證只送一封。
3. P3 的 inhibition 保留 Prometheus 裡的根因與症狀證據，但讓 `WebDown`、`DatabaseDown` 不出現在 firing notifications；最後的 resolved notification 仍可能包含它們。
4. Live 驗證必須鎖定同一個 `run_id`，同時看 Grafana `Alert State` 與 Receiver payload；Notebook 靜態圖不是設定生效證明。
5. P4 是選做：可再練習以 `maintenance` label 與 severity routing 決定 receiver。上游沒有 context label 的臨時例外則可用 Silence API／`amtool`。

完成 P1–P3 後，最後的問題不是「detector 有沒有抓到」，而是：**這個 signal 是否需要一個人現在採取行動？**
